<div dir='rtl'>
    Import کردن کتابخانه های مورد نیاز
</div>

In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


<div dir='rtl'>
    مرتب کردن و آماده سازی دیتافریم 
</div>

In [3]:
movies['year'] = movies['title'].str.extract(r'(\d\d\d\d)')
movies['title'] = movies['title'].str.extract(r'^(.*?)\s*\(')
movies.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [119]:
ratings.head()

,userId,movieId,rating,time
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51


In [5]:
ratings['time'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings.drop(columns='timestamp', inplace=True)

<div dir='rtl'>
    قبل از اضافه کردن داده های کاربر جدید ستون فیلمهایی که دوست داره رو برمیگردونم تا movie id رو هم داشته باشم
</div>

In [6]:
movies.loc[movies['title'].isin(['Titanic', 'Toy Story', 'Inception', 'Interstellar'])]

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1291,1721,Titanic,Drama|Romance,1997
2543,3404,Titanic,Action|Drama,1953
7372,79132,Inception,Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,2010
8376,109487,Interstellar,Sci-Fi|IMAX,2014


In [16]:
user_input = pd.DataFrame({
    'movieId': [1, 1721, 79132, 109487],
    'title': ['Titanic', 'Toy Story', 'Inception', 'Interstellar'],
    'rating': [4.5, 4.8, 5, 5]
})
user_input

,movieId,title,rating
0,1,Titanic,4.5
1,1721,Toy Story,4.8
2,79132,Inception,5.0
3,109487,Interstellar,5.0


<div dir='rtl'>
    مرحله بعد گروه بندی کردن افرادی است که حداقل یکی از فیلم های یوزر مارو دیده و بهش امتیاز داده
</div>

In [17]:
simiular_user = ratings.loc[ratings['movieId'].isin(user_input['movieId'].to_list())]
simiular_user.shape

(571, 4)

<div dir='rtl'>
    سپس انتخاب یوزر هایی که هر 4 تا فیلم یوزر مارو دیده اند
</div>

In [18]:
top = simiular_user[simiular_user.groupby('userId')['rating'].transform('count') == 4]
top

,userId,movieId,rating,time
1772,18,1,3.5,2016-02-11 16:56:56
1899,18,1721,4.0,2016-02-17 22:34:03
2162,18,79132,4.5,2016-02-09 20:34:19
2217,18,109487,4.5,2016-02-09 20:34:47
3219,21,1,3.5,2014-08-09 21:14:38
...,...,...,...,...
95947,601,109487,5.0,2015-09-07 15:19:44
99534,610,1,5.0,2016-11-19 08:08:20
99654,610,1721,3.0,2016-11-19 08:28:32
100452,610,79132,4.0,2017-05-03 20:51:43


<div dir='rtl'>
    اجرای فرمول Pearson برای شناخت شباهت های رفتاریه هرکابر به کاربر ورودی
</div>

In [90]:
from math import sqrt

pearsonCorrelationDict = {}

subgroup = top.groupby(['userId'])
for name, group in subgroup:
    group.sort_values(by='movieId', inplace=True)
    user_input.sort_values(by='movieId', inplace=True)

    temp_user_rating = user_input['rating'].to_list()
    temp_group_rating = group['rating'].to_list()
    ngroups = len(temp_group_rating)

    Sxx = sum([i**2 for i in temp_user_rating]) - pow(sum(temp_user_rating), 2) / float(ngroups)
    Syy = sum([i**2 for i in temp_group_rating]) - pow(sum(temp_group_rating), 2) / float(ngroups)
    Sxy = sum([i*j for i, j in zip(temp_group_rating, temp_user_rating)]) - sum(temp_group_rating) * sum(temp_user_rating) / float(ngroups)

    if Sxx != 0 and Syy != 0:
        pearsonCorrelationDict[name] = Sxy/sqrt(Sxx*Syy)
    else:
        pearsonCorrelationDict[name] = 0

print(pearsonCorrelationDict)

{(18,): 0.9945577827231193, (21,): -0.49374193110106596, (50,): -0.8551861104941836, (233,): 0.36650833306890607, (249,): 0.39900373444305687, (339,): 0.8551861104941488, (414,): 0.7735449421179771, (483,): 0.39900373444305687, (514,): 0, (534,): -0.03683547343419736, (560,): 0.47886115464444007, (561,): -0.6998739952495824, (573,): 0.07053456158586657, (599,): 0.4937419311010259, (601,): 0.8551861104941488, (610,): -0.6401625593124349}


<div dir='rtl'>
    مرتب سازی دیتافریم همبستگی که بدست آوردیم و ترتیب بر اساس مثبت ترین همبستگی
</div>

In [126]:
pearsonDF = pd.DataFrame.from_dict(pearsonCorrelationDict, orient='index', columns=['pearson correlation'])
pearsonDF['userId'] = pearsonDF.index
pearsonDF.index = range(len(pearsonDF))
pearsonDF.sort_values(by='pearson correlation', ascending=False, inplace=True)
pearsonDF['userId'] = pearsonDF.iloc[:, 1].apply(lambda x : x[0])

pearsonDF.head()

,pearson correlation,userId
0,0.994558,18
5,0.855186,339
14,0.855186,601
6,0.773545,414
13,0.493742,599


<div dir='rtl'>
    انجام عملیات merge برای پیدا کردن باقی فیلم های مشاهده شده توسط کاربرانی که بیشترین همبستگی رو دارند
</div>

In [146]:
topUsersRating = pearsonDF.merge(ratings, left_on='userId', right_on='userId', how='inner')
topUsersRating.sample()

,pearson correlation,userId,movieId,rating,time
9565,-0.036835,534,73804,4.0,2016-04-04 18:10:05


<div dir='rtl'>
    اضافه کردن ستون weight که حاصل ضرب مقدار همبستگی و امتیاز فیلم هستش
</div>

In [147]:
topUsersRating['weight'] = topUsersRating['pearson correlation'] * topUsersRating['rating']
topUsersRating.drop(columns='time', inplace=True)
topUsersRating.head()

,pearson correlation,userId,movieId,rating,weight
0,0.994558,18,1,3.5,3.480952
1,0.994558,18,2,3.0,2.983673
2,0.994558,18,6,4.0,3.978231
3,0.994558,18,16,4.5,4.475510
4,0.994558,18,32,4.0,3.978231


In [150]:
tempTopUsersRating = topUsersRating.groupby('movieId').sum()[['pearson correlation','weight']]
tempTopUsersRating.columns = ['sum_pearson correlation','sum_weight']
tempTopUsersRating.head()

,sum_pearson correlation,sum_weight
movieId,,
1,2.960328,10.537352
2,1.899935,5.354746
3,1.267287,3.834793
5,0.472675,0.444977
6,1.847529,6.258546


<div dir='rtl'>
    ایجاد Recommedation matrix و پیشنهاد فیلم به کاربر جدید
</div>

In [158]:
recommendationMatrix = pd.DataFrame()
recommendationMatrix['weighted average recommendation score'] = tempTopUsersRating['sum_weight'] / tempTopUsersRating['sum_pearson correlation']
recommendationMatrix['movie id'] = tempTopUsersRating.index
recommendationMatrix.sort_values(by='weighted average recommendation score', ascending=False, inplace=True)

recommendationMatrix.head()

,weighted average recommendation score,movie id
movieId,,
7573,3.079801e+13,7573
7570,2.463840e+13,7570
3984,2.463840e+13,3984
4005,1.847880e+13,4005
4700,1.231920e+13,4700


In [170]:
suggestedMovies = movies.loc[movies['movieId'].isin(recommendationMatrix['movie id']).to_list()].head(10)
suggestedMovies

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995
5,6,Heat,Action|Crime|Thriller,1995
6,7,Sabrina,Comedy|Romance,1995
7,8,Tom and Huck,Adventure|Children,1995
8,9,Sudden Death,Action,1995
9,10,GoldenEye,Action|Adventure|Thriller,1995
10,11,"American President, The",Comedy|Drama|Romance,1995
